In [1]:
# %pip install mplfinance

In [2]:
import pandas as pd
import os

def read_taifex(path:str, product: str, product_m_or_w: str):
    '''
        讀取台灣期貨交易所的歷史資料，並選取特定商品代號及月份(週別)的資料

        年月日: str 
        商品代號: str
        商品月份: str

        回傳: DataFrame
    '''
    
    # 讀取 CSV 檔案，並將所有元素轉換成字串
    if os.path.exists(path):
        df = pd.read_csv(path, encoding="BIG5", dtype=str)
    else:
        return pd.DataFrame()
     
    # 移除所有元素的空白
    df = df.map(lambda x: x.strip())

    # 當 '到期月份(週別)' 包含 '/' 時，將 '成交價格' 的值改為 '近月價格' 的值
    df.loc[df['到期月份(週別)'].str.contains('/'), '成交價格'] = df['近月價格']
    
    # 到期月份(週別)中有些資料是202404/202405，代表轉倉，因此分割字串並選取第一個元素並存到新欄位 "商品月份"
    df['商品月份'] = df['到期月份(週別)'].str.split('/').str[0]

    df = df[df["商品代號"] == product]
    df = df[df["商品月份"] == product_m_or_w]

    # 將"交易日期" 和 "交易時間" 合併成一個新的欄位 "交易時間"
    df['Date'] = df['成交日期'] + df['成交時間']

    # 然後，將 "交易時間" 轉換成 datetime 對象
    # 假設 "交易日期" 是 "YYYYMMDD" 的格式，"交易時間" 是 "HHMMSS" 的格式
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d%H%M%S')

    df.set_index('Date', inplace=True)
    df = df.drop(columns=['成交日期'])
    df = df.drop(columns=['成交時間'])

    return df

In [3]:
import pandas as pd
import os
from datetime import datetime, time
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import mplfinance as mpf

# 假設你的 DataFrame 是 df
def to_kline(interval:str, df:pd.DataFrame):
    '''
    將原始交易資料轉換為 K 線資料

    參數:
        interval: K 線的時間間隔，例如 '5min' 代表 5 分鐘
        df: 原始交易資料，必須包含 '成交數量(B+S)' 和 '成交價格' 兩個欄位

    回傳值
        df_kline: K 線資料，包含 'Open', 'High', 'Low', 'Close', 'Volume' 五個欄位
    '''

    df.index = pd.to_datetime(df.index)  # 確保索引是 datetime 對象
    df['成交數量(B+S)'] = df['成交數量(B+S)'].astype(int)
    df['成交價格'] = df['成交價格'].astype(float)

    # 計算 OHLC
    df_ohlc = df['成交價格'].resample(interval).agg({'Open':'first', 'High':'max', 'Low':'min', 'Close':'last'})

    # 計算 Volume
    df_volume = df['成交數量(B+S)'].resample(interval).sum()

    # 將兩個 DataFrame 合併
    df_kline = pd.concat([df_ohlc, df_volume], axis=1)
    df_kline = df_kline.rename(columns={'成交數量(B+S)': 'Volume'})  # 將 '成交數量(B+S)' 重新命名為 'Volume'

    return df_kline

def fetch_time(datetime:datetime, time_start:time=time(8,45), time_end:time=time(13,45), interval:str="5min", symbol:str="MTX"):
    year = datetime.strftime('%Y')
    month = datetime.strftime('%m')
    day = datetime.strftime('%d')

    path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'taifex', f'Daily_{year}_{month}_{day}.csv')
    
    # 讀取檔案，如果當天的資料不存在，則讀取下一個月的資料(轉倉後的資料)
    df = read_taifex(path, symbol, year + month)
    if df.empty:
        df = read_taifex(path, symbol, year + (datetime + relativedelta(months=1)).strftime('%m'))
    if df.empty:
        return None
    df_kline = to_kline(interval, df)

    # 轉成固定時間區間
    # df_kline.index = pd.to_datetime(df_kline.index)
    iday = df_kline[df_kline.index.date == datetime.date()]
    iday = iday[iday.index.time >= time_start]
    iday = iday[iday.index.time < time_end]

    return iday

def draw_kline(iday:pd.DataFrame):
    '''
        繪製 K 線圖
        參數:
            iday : pd.DataFrame : 股票資料
        回傳:
            kline_path : str : 圖片檔案路徑
    '''

    mc = mpf.make_marketcolors(up='r',down='g',inherit=True)
    s  = mpf.make_mpf_style(base_mpf_style='tradingview',marketcolors=mc)
    if int(iday.shape[0]) == 0:
        return None, None
    
    # 繪製 K 線圖
    fig, axes = mpf.plot(iday.dropna(), type='candle', style=s,
            ylabel='price',
            volume=True,
            returnfig=True)
    plt.close()

    kline_path = os.path.join("img", "fig.png")
    fig.savefig(kline_path)

    return kline_path

In [4]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def kline_response(kline_path):
    # todo : 這邊要改成從資料庫讀取資料

    if kline_path is None:
        return None

    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", 
                                google_api_key = os.getenv('GEMINI_API_KEY'),)
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": "這是一張小台指的5分k線圖，請根據你對於技術型態的知識，仔細說明這張圖的具體形態，關鍵價格，成交量的變化趨勢，並猜測其隱含意義",
            },  # You can optionally provide text parts
            {"type": "image_url", "image_url": kline_path},
        ]
    )

    response_kline = llm.invoke([message])
    return response_kline.content

In [6]:
def sig_response(llm, response_kline):
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": f'''
                你是一個厲害的操盤手，是當日沖銷的高手，請透過{response_kline}深入分析接下來一小時應該要作多或是作空，你只能回答 #long 或  #short 或 #unknown。
                ''',
            }, 
        ]
    )

    response = llm.invoke([message])
    return response.content

In [8]:
# k 線圖

from datetime import datetime, time
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import timedelta
import json

# 創建日期範圍
start_date = "2023-08-01"
end_date = "2023-12-31"
date_range = pd.date_range(start=start_date, end=end_date)

# 創建時間範圍  9:00 ~ 13:00
time_range = [time(hour) for hour in range(9,14)]

# 對於每一天的每一小時調用 fetch_time
for date in date_range:
    for start_time in time_range:
        end_time = (datetime.combine(date, start_time) + timedelta(hours=1)).time()
        date_str = date.strftime("%Y%m%d")
        start_time_str = start_time.strftime("%H")
        end_time_str = end_time.strftime("%H")
        formatted_str = f"{date_str}_{start_time_str}_{end_time_str}"
        print(f"date: {date_str}, 從：{start_time_str}～{end_time_str}")
        df = fetch_time(date, start_time, end_time, "5min", "MTX")
        if df is None or len(df)<1:
            continue
        # df.to_csv("temp_data/" + formatted_str + ".csv")
        change = df['Close'].iloc[-1]/df['Open'].iloc[0]
        
        path = draw_kline(df)
        res = kline_response(path)
        sig_str = sig_response(llm, res)
        if sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        print(sig_str, sig)

        output_data = {
            "date": date_str,
            "start_time": start_time_str,
            "end_time":end_time_str,
            "model": model,
            "temperature": temperature,
            "top_p": top_p,
            "seed": seed,
            "response_sig": sig,
            "change": change,
            "sig_str":sig_str,
        }
        json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
        out_path = os.path.join("out_kline")
        os.makedirs(out_path, exist_ok=True)
        with open(out_path + "/" + formatted_str + ".json", 'w', encoding='utf-8') as f:
            f.write(json_content)
        


date: 20230801, 從：09～10
#unknown 
 0
date: 20230801, 從：10～11
#short 

根據圖表顯示，多方力量在10:30後開始減弱，價格出現回落，並且成交量也持續減少，顯示市場觀望情緒加重。 17150是重要的阻力位，价格已跌破此水平，預計短期內會繼續下跌。
 -1
date: 20230801, 從：11～12
#short 
 -1
date: 20230801, 從：12～13
#unknown 
 0
date: 20230801, 從：13～14
根據你提供的資訊，這張圖顯示小台指在5分鐘K線圖上呈現看漲趨勢，但市場參與者積極性有所下降。

* 多頭吞噬形態和價格在回調過程中沒有跌破17180點支撐位，表明市場看漲情緒仍然較為強勁。
* 成交量逐漸減少，顯示市場參與者積極性下降，未來價格走勢需要關注成交量的變化。
* 價格是否能突破17200點阻力位將會是關鍵。

**由於目前資訊不足，無法確定接下來一小時應該要作多或是作空。**

**#unknown** 
 0
date: 20230802, 從：09～10
#long 
 1
date: 20230802, 從：10～11
#short 
 -1
date: 20230802, 從：11～12


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


#short 
 -1
date: 20230802, 從：12～13
#unknown 

雖然圖表顯示一些看跌訊號，例如吞噬形態和十字星，但突破也暗示可能出現反彈。成交量的變化也表明市場情緒不穩定。 

作為一個負責任的 AI，我無法提供具體的交易建議。市場走勢受到各種因素影響，包括新聞、經濟數據和市場情緒。 

我建議您進行更深入的研究，並考慮使用其他技術指標來支持您的交易決策。
 0
date: 20230802, 從：13～14
#long 
 1
date: 20230803, 從：09～10
date: 20230803, 從：10～11
date: 20230803, 從：11～12
date: 20230803, 從：12～13
date: 20230803, 從：13～14
date: 20230804, 從：09～10
#short 
 -1
date: 20230804, 從：10～11
#unknown 
 0
date: 20230804, 從：11～12
#short 
 -1
date: 20230804, 從：12～13
#unknown 

雖然圖表顯示短線多頭訊號，但成交量低且價格在關鍵阻力位附近，無法確定接下來一小時的走勢。 需要觀察更多資訊，例如: 

* 成交量的變化
* 突破16830的力度
* 其他技術指標的訊號

建議在市場趨勢不明朗時，謹慎操作，避免過度冒險。 
 0
date: 20230804, 從：13～14
#short 
 -1
date: 20230805, 從：09～10
date: 20230805, 從：10～11
date: 20230805, 從：11～12
date: 20230805, 從：12～13
date: 20230805, 從：13～14
date: 20230806, 從：09～10
date: 20230806, 從：10～11
date: 20230806, 從：11～12
date: 20230806, 從：12～13
date: 20230806, 從：13～14
date: 20230807, 從：09～10
#unknown 
 0
date: 20230807, 從：10～11
#long 
 1
date: 20230807, 從：11～12
#unknown 

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


#short 
 -1
date: 20230808, 從：10～11
#unknown 
 0
date: 20230808, 從：11～12


...

#unknown 
 0
date: 20231130, 從：13～14
#short 
 -1
date: 20231201, 從：09～10
#long 
 1
date: 20231201, 從：10～11
#short 
 -1
date: 20231201, 從：11～12
#unknown 
 0
date: 20231201, 從：12～13
#unknown 
 0
date: 20231201, 從：13～14
#long 
 1
date: 20231202, 從：09～10
date: 20231202, 從：10～11
date: 20231202, 從：11～12
date: 20231202, 從：12～13
date: 20231202, 從：13～14
date: 20231203, 從：09～10
date: 20231203, 從：10～11
date: 20231203, 從：11～12
date: 20231203, 從：12～13
date: 20231203, 從：13～14
date: 20231204, 從：09～10
#long 
 1
date: 20231204, 從：10～11
#unknown 

虽然图表显示了看涨迹象，但价格回落，成交量也下降，这表明买盘力量正在减弱。 突破阻力位 17410 点或跌破支撑位 17380 点，才能确定接下来的趋势。 
 0
date: 20231204, 從：11～12
#unknown 
 0
date: 20231204, 從：12～13
#long 
 1
date: 20231204, 從：13～14
#short 
 -1
date: 20231205, 從：09～10
#unknown 
 0
date: 20231205, 從：10～11
#short 
 -1
date: 20231205, 從：11～12
#short 
 -1
date: 20231205, 從：12～13
#unknown 

由於提供的資訊只是一個抽象的描述，沒有實際的圖表可以分析，無法進行技術分析判斷接下來一小時應該要作多或是作空。 
 0
date: 20231205, 從：13～14
#long 
 1
date: 20231206, 從：09～10
#long 
 1
dat

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


#short 
 -1
date: 20231208, 從：10～11
#long 
 1
date: 20231208, 從：11～12
#short 
 -1
date: 20231208, 從：12～13
#short 
 -1
date: 20231208, 從：13～14
#unknown 

雖然圖表顯示短線反彈的可能性，但我需要更多資訊才能決定接下來一小時應該做多或做空。例如：

* **更長時間的趨勢：** 小台指在過去幾天的走勢如何？是整體上升還是下降趨勢？
* **其他技術指標：** 除了價格和成交量，還有其他技術指標可以提供更多資訊，例如移動平均線、RSI、KD 等。
* **市場消息面：** 任何新聞或事件會影響市場情緒？例如經濟數據、政策公告等。

只有綜合以上資訊才能做出更準確的判斷。 
 0
date: 20231209, 從：09～10
date: 20231209, 從：10～11
date: 20231209, 從：11～12
date: 20231209, 從：12～13
date: 20231209, 從：13～14
date: 20231210, 從：09～10
date: 20231210, 從：10～11
date: 20231210, 從：11～12
date: 20231210, 從：12～13
date: 20231210, 從：13～14
date: 20231211, 從：09～10
#long 
 1
date: 20231211, 從：10～11
#short 
 -1
date: 20231211, 從：11～12
#long 
 1
date: 20231211, 從：12～13
#short 
 -1
date: 20231211, 從：13～14
#long 
 1
date: 20231212, 從：09～10
#long 
 1
date: 20231212, 從：10～11


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


#unknown 

雖然圖表顯示買方力量仍強，但成交量下降可能代表多方力量減弱。這是一個不確定的情況，需要更多資訊才能決定下一步行動。 

以下因素需要考慮：

* **新聞事件:** 任何重大新聞事件可能會影響市場情緒，例如財經數據或政策公告。
* **技術指標:** 其他技術指標可以提供更多關於市場趨勢的信息，例如移動平均線或RSI。
* **個人風險承受能力:**  投資者需要根據自己的風險承受能力和投資策略做出投資決策。

建議密切關注市場動態，並根據最新信息調整投資策略。 
 0
date: 20231212, 從：11～12
#unknown 

雖然圖表顯示了買賣力量的變化，但僅憑一個小時的數據還不足以做出確定的交易決策。你需要更全面的分析，例如：

* **更長期的趨勢:** 小台指的日線、周線、月線圖走勢如何？
* **其他技術指標:**  例如MACD、RSI等，是否有明顯的買賣訊號？
* **市場消息面:** 近期是否有重大事件或消息會影響市場走勢？
* **個人風險偏好:** 你願意承擔多少風險？

僅憑一張圖表做出交易決定非常危險，建議你進行更深入的研究，並结合其他因素做出更理性的判断。
 0
date: 20231212, 從：12～13
#short 

雖然圖表顯示可能有反彈的跡象，但整體趨勢仍然向下，成交量也持續下降。在這種情況下，應該保持謹慎，等待更多確認訊號，例如突破下降通道或成交量明顯增加。
 -1
date: 20231212, 從：13～14
#long 
 1
date: 20231213, 從：09～10
date: 20231213, 從：10～11
date: 20231213, 從：11～12
date: 20231213, 從：12～13
date: 20231213, 從：13～14
date: 20231214, 從：09～10
date: 20231214, 從：10～11
date: 20231214, 從：11～12
date: 20231214, 從：12～13
date: 20231214, 從：13～14
date: 20231215, 從：09～10
date: 20231215, 從：10～11
date: 20231215, 從：11～12
date: 20231215, 從：12～13

In [13]:
import pandas as pd
import json
import os

# 計算準確率和平均報酬率
gain = []
loss = []

tp = 0
fp = 0
tn = 0
fn = 0

files = os.listdir("out_kline")
files = sorted(files)

for idx, file in enumerate(files):
    # if file.split("_")[1] != "09" and file.split("_")[1] != "09":
    #     continue
    # print(file)
    with open("out_kline/" + file, "r", encoding='utf-8') as f:
        data = json.load(f)
        sig_str = data["sig_str"]
        sig = 0
        if sig_str.find("unknown") > 0:
            sig = 0
        elif sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        print(sig, sig_str)
        # sig = data["response_sig"]
        # print(data["response_sig"])

    if idx+1 == len(files):
        break
    # print(files[idx+1])
    with open("out_kline/" + files[idx+1], "r", encoding='utf-8') as f:
        data = json.load(f)
        rtn = data["change"]
        rtn = float(rtn) - 1
        # print(rtn)
    # print()
    
    if pd.isna(rtn):
        rtn = 0

    if sig > 0:
        if rtn > 0:
            tp += 1
            gain.append(rtn)
        else:
            fp += 1
            loss.append(rtn)
    elif sig == -1: 
        if rtn < 0:
            tn += 1
            gain.append(-rtn)
        else:
            fn += 1
            loss.append(-rtn)


accuracy = (tp + tn) / (tp + fp + tn + fn)
print(f'Accuracy: {accuracy}')
print(f'Average gain: {sum(gain) / len(gain)}')
print(f'Average loss: {sum(loss) / len(loss)}')
print(f'expected value: {(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')
# print(tn+tp+fn+fp)

1 #long 

0 #unknown 

0 #unknown 

-1 #short 

1 #long 

0 #unknown 

-1 #short 

-1 #short 

1 #long 

0 #unknown 

-1 #short 

-1 #short 

-1 #short 

0 #unknown 

雖然多頭吞噬形態表明買方力量正在增強，但目前價格尚未突破 17546 的關鍵阻力位。同時，成交量雖然有所增加，但還不足以確認趨勢完全反轉。

建議密切關注接下來的價格走勢，若價格突破 17546 且成交量持續放大，則可以考慮 #long。反之，若價格跌破 17536 且成交量持續萎縮，則可以考慮 #short。

-1 #short 

1 #long 

1 #long 

-1 #short 

1 #long 

1 #long 

1 #long 

-1 #short 

0 #unknown 

-1 #short 

-1 #short 

0 #unknown 

0 #unknown 

-1 #short 

0 #unknown 

雖然圖表顯示了一些下跌趨勢和支撐位的跡象，但僅憑一個小時的數據無法準確預測接下來一小時的走勢。市場變化快速，影響因素眾多，技術分析僅供參考，不應作為唯一交易依據。 

**謹慎投資，風險自負。**

-1 #short 

1 #long 

-1 #short 

-1 #short 

0 #unknown 

0 #unknown 

雖然圖表顯示了一些看漲的跡象，但單憑 40 分鐘的數據不足以做出明確的交易決策。以下是一些需要考慮的因素：

* **吞噬模式的可靠性:** 吞噬模式是一種可靠的反轉模式，但它需要與其他指標和分析相結合才能得到驗證。
* **成交量:** 成交量的增加表明市場參與者的興趣正在增加，但這並不一定意味著價格會反轉向上。
* **整體市場趨勢:** 我們不知道小台指的整體市場趨勢是什麼。如果整體市場趨勢仍然看跌，那麼即使出現吞噬模式，價格也可能繼續下跌。
* **其他指標:** 除了成交量之外，還有許多其他指標可以幫助我們判斷價格走勢，例如移動平均線、RSI、MACD 等。

因此，在做出交易決策之前，需要進行更深入的分析，考慮其他指標和市場因素。

1 